# Insights Lab: Evaluation

Evaluate the exported model using accuracy, confusion matrix, and feature importance.

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

ARTIFACTS_DIR = Path('..') / 'artifacts'
model = joblib.load(ARTIFACTS_DIR / 'placement_pipeline.joblib')
df = pd.read_csv(ARTIFACTS_DIR / 'cleaned_dataset.csv')

feature_cols = ['cgpa', 'internships', 'work_experience', 'branch', 'gender']
df['work_experience'] = df.get('work_experience', df['internships']).fillna(df['internships'])
X = df[feature_cols]
y = df['placed']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preds = model.predict(X_test)
accuracy_score(y_test, preds)

In [ ]:
cm = confusion_matrix(y_test, preds)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
print(classification_report(y_test, preds))

In [ ]:
preprocessor = model.named_steps['preprocess']
classifier = model.named_steps['model']

feature_names = preprocessor.get_feature_names_out()
if hasattr(classifier, 'feature_importances_'):
    importances = classifier.feature_importances_
else:
    importances = abs(classifier.coef_[0])

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

importance_df.head(12)